All packages are installed in a venv with system passthrough, setup your own environment as you like.

In [13]:
# Imports

import numpy as np
import pandas as pd
import torch

# Force pandas to print to 3 decimal places
pd.set_option('display.float_format', lambda x: '%.3f' % x)

Housing dataset is from [here](https://www.kaggle.com/c/house-prices-advanced-regression-techniques), and the macroeconomic home value dataset is from [here](https://www.kaggle.com/datasets/sagarvarandekar/macroeconomic-factors-affecting-us-housing-prices)

In [19]:
# Import housing dataset and macro datasets

df_housing = pd.read_csv('housing.csv')
df_macro = pd.read_csv('macro.csv')

In [20]:
# - Repair housing dataset -

# This is a kaggle competition dataset, so there are plenty of designed issues to fix here

df_housing = df_housing.drop(columns=['Id'])

# Fix categorical NaN -> there is none (i.e. no pool)
categorical_cols = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType'
]
for col in categorical_cols:
    if col in df_housing.columns:
        df_housing[col] = df_housing[col].fillna('None')

# Fix numerical NaN -> missing data, fill w/ median to prevent outlier skew 
numerical_cols = ['LotFrontage', 'MasVnrArea', 'GarageYrBlt']
for col in numerical_cols:
    if col in df_housing.columns:
        df_housing[col] = df_housing[col].fillna(df_housing[col].median())

# Drop cols where target pred. is missing somehow 
df_housing = df_housing.dropna(subset=['SalePrice'])

# Make mergekey using year_month using YrSold and MoSold, arbitrarily set day to 1 for conversion then make monthly
df_housing['year_month'] = pd.to_datetime(
    df_housing[['YrSold', 'MoSold']]
        .assign(DAY=1)
        .rename(columns={'YrSold': 'YEAR', 'MoSold': 'MONTH'})
).dt.to_period('M')

# There is ONE house without it's electrical, because it's categorical I'll just fill it with the mode
df_housing['Electrical'] = df_housing['Electrical'].fillna(df_housing['Electrical'].mode()[0])

# - Repair macro dataset - 

# Convert DATE to datetime, make matching year_month key
df_macro['DATE'] = pd.to_datetime(df_macro['DATE'])
df_macro['year_month'] = df_macro['DATE'].dt.to_period('M')

# Replace missing var with median
macro_missing_cols = ['MED HOUSEHOLD INCOME', '% SHARE OF WORKING POPULATION']
for col in macro_missing_cols:
    df_macro[col] = df_macro[col].fillna(df_macro[col].median())

In [21]:
# - Merge housing and macro datasets -

# Inner join, append economic data to each housing sale only if both match
# The actual overlap is only ~33% between datasets, would destroy analysis if we filled w/ median
df_merged = pd.merge(df_housing, df_macro, on='year_month', how='inner')

# Remove merge key and the duplicated DATE
df_merged = df_merged.drop(columns=['year_month', 'DATE'])

print(df_merged.head())
print(df_merged.info())

   MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0         190       RL       50.000     7420   Pave  None      Reg   
1         190       RL       50.000     7420   Pave  None      Reg   
2         190       RL       50.000     7420   Pave  None      Reg   
3         190       RL       50.000     7420   Pave  None      Reg   
4         190       RL       50.000     7420   Pave  None      Reg   

  LandContour Utilities LotConfig  ... INFLATION(%)  \
0         Lvl    AllPub    Corner  ...        0.091   
1         Lvl    AllPub    Corner  ...        1.070   
2         Lvl    AllPub    Corner  ...        3.655   
3         Lvl    AllPub    Corner  ...        4.937   
4         Lvl    AllPub    Corner  ...        5.372   

  MORTGAGE INT. MONTHLY AVG(%) MED HOUSEHOLD INCOME CORP. BOND YIELD(%)  \
0                        5.286            50303.000               5.050   
1                        6.088            50303.000               6.120   
2                      

In [22]:
# Stats (mean, std, min, max)
print(df_merged.describe())

# Manually do var and med
num_col = df_merged.select_dtypes(include=['float', 'int'])
print()
print(f"Variance:\n{num_col.var()}")
print()
print(f"Median:\n{num_col.median()}")
print()

# Group saleprice by neighborhood
grp = df_merged.groupby("Neighborhood")['SalePrice'].agg(['mean', 'median', 'std', 'count'])

# Print sorted by median
print("\n--- Sale Price by Neighborhood ---")
print(grp.sort_values(by='median', ascending=False))

       MSSubClass  LotFrontage   LotArea  OverallQual  OverallCond  YearBuilt  \
count     696.000      696.000   696.000      696.000      696.000    696.000   
mean       56.293       74.690 12237.672        5.862        5.276   1969.621   
std        47.953       37.161  9068.115        1.853        0.980     31.029   
min        20.000       21.000  2001.000        1.000        3.000   1900.000   
25%        20.000       60.000  7742.000        5.000        5.000   1949.000   
50%        47.500       69.000 10383.000        6.000        5.000   1973.000   
75%        60.000       80.000 13704.000        7.000        6.000   2001.000   
max       190.000      313.000 63887.000       10.000        7.000   2008.000   

       YearRemodAdd  MasVnrArea  BsmtFinSF1  BsmtFinSF2  ...  INFLATION(%)  \
count       696.000     696.000     696.000     696.000  ...       696.000   
mean       1981.983     127.931     588.931      27.966  ...         2.273   
std          22.879     229.678     

In [ ]:
# TODO: data vis

In [ ]:
# TODO: ML model 1

In [ ]:
# TODO: ML model 2 or merge other dataset into this housing one